# RedActa — локальное демо

Запускается в **локальном Jupyter** (не Colab). Требования:
- Python-зависимости из `requirements.txt` установлены в активном окружении
- Ollama запущен локально (`ollama serve`) и модель уже загружена

Поддерживаемые операции:
- `replace_point` — изложить пункт в новой редакции
- `repeal_point` — признать пункт утратившим силу
- `insert_point` — вставить новый пункт
- `append_section_item` — дополнить подпункт или элемент
- `replace_phrase_globally` — заменить или удалить фразу по документу
- `replace_appendix_block` — изложить в новой редакции структурно однозначный блок приложения

In [ ]:
# ── Ячейка 1: Пути к файлам — отредактируйте перед запуском ───────────────
from pathlib import Path

# Корень репозитория: автоопределение (Jupyter запущен из корня или из notebooks/)
def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate.resolve()
    raise RuntimeError(
        "Не удалось найти корень репозитория. "
        "Запустите Jupyter из папки проекта или раскомментируйте REPO_DIR ниже."
    )

REPO_DIR = _find_repo_root()
# REPO_DIR = Path(r"C:\path\to\RedActa-main")  # альтернатива: задать явно

# Пути к документам — укажите свои файлы
BASE_DOCX      = REPO_DIR / "redactions" / "demo" / "base.docx"
AMENDMENT_DOCX = REPO_DIR / "redactions" / "demo" / "amendment.docx"

# Конфиг модели
MODELS_CONFIG = REPO_DIR / "config" / "models.colab.json"

# ID кейса — определяет папку вывода artifacts/<CASE_ID>/
CASE_ID = "demo"

# Проверка
for label, p in [("base", BASE_DOCX), ("amendment", AMENDMENT_DOCX), ("config", MODELS_CONFIG)]:
    status = "OK" if p.exists() else "НЕ НАЙДЕН"
    print(f"{label:12s}: {p}  [{status}]")

In [ ]:
# ── Ячейка 2: Python path ──────────────────────────────────────────────────
import sys

src_dir = str(REPO_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print(f"src: {src_dir}")

In [ ]:
# ── Ячейка 3: Проверка Ollama ──────────────────────────────────────────────
import urllib.request
import json as _json

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as resp:
        tags = _json.loads(resp.read())
    models = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama готов. Загружено моделей: {len(models)}")
    for m in models:
        print(f"  - {m}")
except Exception as exc:
    print(f"Ollama недоступен: {exc}")
    print("Запустите в терминале: ollama serve")

In [ ]:
# ── Ячейка 4: Запуск пайплайна ────────────────────────────────────────────
import json
from graph_pipeline.colab_runner import run_uploaded_pair

result = run_uploaded_pair(
    base_docx=BASE_DOCX,
    amendment_docx=AMENDMENT_DOCX,
    workspace_root=REPO_DIR,
    models_config=MODELS_CONFIG,
    case_id=CASE_ID,
)

validation = result.get("validation", {})
print("\nСводка валидации:")
print(json.dumps(validation, ensure_ascii=False, indent=2))

output_doc = result.get("output_doc")
print(f"\nВыходной документ: {output_doc}")

status = "valid" if validation.get("is_valid") else "invalid"
print(f"Статус: {status}")

total_markers = sum(len(step.get("revision_markers", [])) for step in result.get("steps", []))
print(f"Маркеры редакции: {total_markers} вставлено")

In [ ]:
# ── Ячейка 5: Открыть выходной документ (Windows / macOS / Linux) ──────────
import subprocess
import platform

if output_doc:
    p = Path(output_doc)
    if p.exists():
        system = platform.system()
        if system == "Windows":
            subprocess.Popen(["start", "", str(p)], shell=True)
        elif system == "Darwin":
            subprocess.Popen(["open", str(p)])
        else:
            subprocess.Popen(["xdg-open", str(p)])
        print(f"Открываю: {p}")
    else:
        print(f"Файл не найден: {p}")
else:
    print("Сначала запустите ячейку 4.")

## Диагностика

- *Файл НЕ НАЙДЕН в ячейке 1* — укажите правильные пути к `.docx` в переменных `BASE_DOCX` / `AMENDMENT_DOCX`.
- *Не удалось найти корень репозитория* — раскомментируйте строку `REPO_DIR = Path(...)` и задайте путь явно.
- *Ollama недоступен (ячейка 3)* — запустите `ollama serve` в отдельном терминале.
- *`ModuleNotFoundError: graph_pipeline`* — убедитесь, что ячейка 2 выполнена и `REPO_DIR` указывает на корень репозитория.
- *Статус `invalid`* — смотрите сводку валидации и папку `artifacts/<CASE_ID>/` с debug JSON.
- *Маркеры редакции: 0* — пайплайн не нашёл однозначных точек привязки; это норма при частичном разрешении intents.